# 原时间 NLL + KL：single case

与 `neural_only/testdata2.py` 共用原始数据、CPU seed 和前80%/后20%划分。
树支持域为 `[0, 1.1 * max_observation]`，最大值仅取训练集有限 L/R 端点。
叶内原时间密度均匀，不使用非线性归一化。默认加载已冻结的 raw_time_finite_kl 选参。
seed123 用于 HPO，本 notebook 是案例展示，不作为新的独立性能评估。


In [ ]:
from pathlib import Path
import sys
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'pycode'/'neural_only').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run inside PolyaPFN')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pycode.kl_single_case.case_study import (
    CaseConfig, generate_case, load_selected, prepare_output, train_case,
    evaluate_case, save_case, plot_case,
)
from pycode.unified_nll.train import TrainingConfig
from pycode.unified_nll.nll_cv_train import NLLCVOptimization
from pycode.unified_nll.tuning_model import Architecture

SEED = 123
N_SAMPLES = 5000
X_DIM = 6
SCENARIO = 'ph'             # ph, lognormal, po, multimodal
CENSOR_TYPE = 'right'       # exact, right, interval, current
CENSOR_Q = 0.5
MULTIMODAL_CASE = 'separate' # separate, closer
GRID_SIZE = 256
LOAD_SAVED_SELECTION = True
INDEX = 4
CPU_THREADS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(CPU_THREADS)
CASE = CaseConfig(seed=SEED, n_samples=N_SAMPLES, x_dim=X_DIM, scenario=SCENARIO,
    censor_type=CENSOR_TYPE, censor_q=CENSOR_Q, multimodal_case=MULTIMODAL_CASE,
    grid_size=GRID_SIZE)
OUTPUT = ROOT/'pycode'/'kl_single_case'/'results'/f'{CASE.setting}_seed{SEED}_raw_time_finite_v1'

# Used when LOAD_SAVED_SELECTION=False. Set these before evaluating test SSE.
ARCHITECTURE = Architecture(depth=3, shared_width=32, shared_layers=1,
    private_width=32, private_layers=3, regularization_strength=500.0, x_dim=X_DIM)
REFIT_EPOCHS = 100
TRAINING = TrainingConfig(epochs=200, batch_size=256, lr=1e-3, patience=50,
    validation_every=5, min_delta=1e-4, max_grad_norm=1.0, cuda_graphs=True)
OPTIMIZATION = NLLCVOptimization(split_lr_ratio=0.1, warmup_epochs=10)
print('Device:', DEVICE, 'Setting:', CASE.setting, 'Output:', OUTPUT)

## Generate exactly the shared example data

直接调用原生成器，生成前重置 CPU seed。训练/测试之间没有额外抽样或重排。

In [ ]:
data = generate_case(CASE)
x, L, R, event_time = data['raw'][:4]
x_train, L_train, R_train = data['train']
x_test, L_test, R_test = data['test']
grid, true_survival = data['grid'], data['truth']
max_obs = data['max_obs']
upper_bound = data['upper_bound']
print('Train/test:', len(x_train), len(x_test), 'max_obs:', max_obs, 'upper_bound:', upper_bound)
print('Observed right-censoring fraction:', float(torch.isinf(R).float().mean()))
plt.figure(figsize=(6, 3))
plt.boxplot(event_time.numpy())
plt.ylabel('event time')
plt.show()

## Fit one KL model

默认参数和轮数来自未加正则验证 NLL 选择的四折 CV；本案例测试数据不传给训练器。
`a/sqrt(n_fit)` 乘深度归一化的叶宽 KL，时间变换与已保存 KL 实验相同。

In [ ]:
if LOAD_SAVED_SELECTION:
    ARCHITECTURE, REFIT_EPOCHS = load_selected(CASE)
print('Architecture:', ARCHITECTURE)
print('Frozen refit epochs:', REFIT_EPOCHS)
manifest = prepare_output(OUTPUT, CASE, ARCHITECTURE, TRAINING, OPTIMIZATION,
                          REFIT_EPOCHS, DEVICE)
model, history, fit_seconds = train_case(data, CASE, ARCHITECTURE, REFIT_EPOCHS,
    training=TRAINING, optimization=OPTIMIZATION, device=DEVICE)
print('Fit seconds:', fit_seconds)

## Evaluate and save

SSE = 256点平方误差求和后，对测试对象取平均，与 neural_only 相同。
本例不通过测试分数重新选择架构、正则参数或停止轮数。

In [ ]:
results_df, estimated, grid_sse = evaluate_case(model, data, fit_seconds)
save_case(OUTPUT, CASE, data, model, history, results_df, estimated, grid_sse)
display(results_df)
print('Saved:', OUTPUT)

In [ ]:
figures = plot_case(model, data, estimated, history, index=INDEX, output=OUTPUT)
print(f'Test observation {INDEX}: grid SSE = {grid_sse[INDEX].item():.6f}')
plt.show()

In [ ]:
grid_sse_df = pd.DataFrame({'NLL + KL': grid_sse.numpy()})
grid_sse_df.index.name = 'test_observation'
display(grid_sse_df.describe())